# Finding similar items
The aim is to detect similar textual items in the `text` field of the Kaggle [Yelp](https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset) dataset.

In [2]:
import os
import json
import pandas as pd
import pip
import string

def import_or_install(package):
    try:
        __import__(package)
    except ImportError:
        pip.main(['install', package])

/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


In [3]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

import findspark
findspark.init("spark-3.5.0-bin-hadoop3")# SPARK_HOME
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

sc = spark.sparkContext

We first of all import the Yelp dataset from Kaggle, using a token.

In [4]:
os.environ['KAGGLE_USERNAME'] = "luciaannamellini"
os.environ['KAGGLE_KEY'] = "c209fcf223ecdd6be8fd373196354f4b"
!kaggle datasets download -d yelp-dataset/yelp-dataset

yelp-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [5]:
from tqdm import tqdm
import zipfile

DATA_DIR = "/content/yelp-dataset"

with zipfile.ZipFile(DATA_DIR + ".zip","r") as zip_ref:
     for file in tqdm(iterable=zip_ref.namelist(), total=len(zip_ref.namelist())):
          zip_ref.extract(member=file, path=DATA_DIR)

100%|██████████| 6/6 [02:08<00:00, 21.38s/it]


We take into consideration the portion of the dataset regarding reviews, contained in `yelp_academic_dataset_review.json`. We convert the resulting dataframe in RDD form.

In [6]:
df_reviews = spark.read.json("/content/yelp-dataset/yelp_academic_dataset_review.json")
reviews_RDD = df_reviews.rdd

We sample the dataset at the only scope to speed up the following steps in the (free) Google Colab environment.

In [7]:
reviews_RDD = reviews_RDD.sample(False, 0.0001, 42)
sample_dim = reviews_RDD.count()

Seen the aim of the project we will only be looking at the `text` attribute of the imported reviews.

In [8]:
text_RDD = reviews_RDD.map((lambda r: (r[0], r['text'])))

Let's look at some reviews of the text field of the review dataset.

In [9]:
print('---\n')
for text in text_RDD.take(4):
    print('{}\n'.format(text[1]))
    print('---\n')

---

Most under-rated brewery on this list. Sun King is garbage. It's just popular from publicity and ads. Granite City has amazing food as well. Keep away from their mixed drinks. Their margaritas are a joke. Downtown location is better than Carmel.

---

My favorite place for Mexican food in Indianapolis! Their meat is full of flavor, their portions are more than enough! It's definitely a carry out place, but it's well worth the wait. Love it here!

---

We called ahead and picked up at the drive through window after leaving the kids ballgame. We ordered 3 kids meals and a poboy. The kids meals came with fries, yogurt, a rice krispy treat and a drink for $5 each. The kids were full, the food was good and it was all for 26 dollars. The shrimp and burgers are both really good. We will be back.

---

This annual June event celebrating Indian jewelry, art and food turns out large crowds. First, the good: Lots of jewelry vendors offer tons of variety, from Santa Fe and New Mexico flavored

## Data pre-processing

We begin by removing text cells that are `None` or that contain empty strings. It is possible to verify that for each review a nonempty text is present.

In [10]:
text_RDD = text_RDD.filter(lambda text: bool(text))

To study the similarity between the texts of the reviews we proceed by looking at the relative string as a set of tokens. We have chosen to divide each review in the terms, all considered in lower case, that compose it. We have preferred this approach against using classical $k$-grams because we are more inteersted in the meaning of the reviews, so we don' care about looking also at the structure of the text.

We get rid of the stop words appearing in the tokens to extract the actual semantics of the text. Also we lemmatize the remaining tokens to consider as similar the various inflections of a certain word. Lastly, for each review we maintain each appearing token once.

In [11]:
%%capture
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stopwords = stopwords.words('english')
import spacy
nlp = spacy.load("en_core_web_sm")

split_regex = r'\W+'

tokenize = lambda string: [s for s in re.split(split_regex,string.lower()) if s not in stopwords and s != '']
text_tokens_RDD = text_RDD.map(lambda s: (s[0],tokenize(s[1])))

lemmatize = lambda word: str([token.lemma_ for token in nlp(word)][0])
text_lemmatized_tokens_RDD = text_tokens_RDD.map(lambda s: (s[0], [lemmatize(w) for w in s[1]]))

text_lemmatized_tokens_RDD = text_lemmatized_tokens_RDD.map(lambda s: (s[0], list(set(s[1]))))

Let's look at how the first of the reviews showed above has been transformed.

In [12]:
text_lemmatized_tokens_RDD.first()

('HCbJPXWXvwN-C7XfmVy3gA',
 ['city',
  'keep',
  'ad',
  'food',
  'location',
  'amazing',
  'margarita',
  'king',
  'popular',
  'joke',
  'sun',
  'rate',
  'list',
  'well',
  'brewery',
  'garbage',
  'downtown',
  'drink',
  'mix',
  'away',
  'carmel',
  'granite',
  'publicity'])

At this point we have codified the text of each review through it's essential information.

## Similar items with Jaccard similarity

We begin by evaluating the similarity of the reviews according to the Jaccard similarity measure between sets. The Jaccard similarity between sets $S$ and $T$ is defined as:
\begin{equation}
J(S,T)=\frac{|S\cap T|}{|S\cup T|}
\end{equation}
In our case the sets have as items the tokens that we have extracted from the reviews.



### Similarity preserving summaries of reviews
To hold the summary of the tokens extracted from each review compactly we consider their characteristic matrix, used to represent a collection of sets.

Though with growing amounts of data it is not realistic to maintain the whole characteristic matrix. So, we construct a succint structure called a signature matrix. Each row of such matrix is constructed as follows:
1. choose a permutation of the rows of the characteristic matrix uniformly at random among all possible permutations,
2. apply the chosen permutation to the rows of the charactistic matrix,
3. apply a minhash function to all columns of the resulting matrix.

A minhash function is defined as $h:\{\text{reviews}\}\to\{\text{shingles}\}$, and for a set of reviews it returns the index of the first one for which a certain token is present.

The column related to a review is it's signature.

We begin by putting aside all the *distinct* tokens found in the dataset.

In [54]:
all_tokens = text_lemmatized_tokens_RDD.flatMap(lambda t: [(s, 1) for s in t[1]]).reduceByKey(lambda a,b: a+b).map(lambda s: (s[0],1))

For each token we want to list the summaries of the reviews in which they appear.


In [55]:
tokens_in_reviews = text_lemmatized_tokens_RDD.flatMap(lambda s: [(t,s[0]) for t in s[1]]).reduceByKey(lambda a,b: list(a[1])+list(b[1]))

In [57]:
tokens_in_reviews.filter(lambda s: len(s[1])>1).first()

('city', 'HCbJPXWXvwN-C7XfmVy3gA')

In [38]:
import math
def row_permutation(rdd):
    return -1

def minhash(set, items):
    set.
    return item_index

def construct_signature_matrix(review_tokens, all_tokens)
    signature_matrix = review_tokens.map(lambda s: (s[0], math.inf))
    signature_matrix = signature_matrix.map(lambda s: (s[0], minhash(s[1],all_tokens)))
    return signature_matrix



signature_matrix = construct_signature_matrix(text_lemmatized_tokens_RDD)

We can use the the signature matrix to estimate the Jaccard similarity between reviews because it is possible to prove that $\mathbb{P}(H(S_1)=H(S_2))=J(S_1,S_2)$, where $H$ is a minhash function applied to a random permutation of the review set. So, by estimating the probability by looking at the relative frequency of having the same signature for some pair of reviews, we obtain their approximate similarity.

### Locality-sensitive hashing (LSH)

But, considering the number of rows in the signature matrix it would be too costly to scan all of them to compute the relative frequency between all possible pairs of reviews. So, we proceed by applying Locality-Sensitive Hashing. In this approach we filter on pairs of reviews by hashing them several times and only looking at those couples collected in the same bucket. The rationale is that similar reviews are more likely to be hashed in the same bucket, so we hope that dissimilar pairs end up in distinct buckets, and thus are never checked for similarity.

In [37]:
signature_matrix.first()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "spark-3.5.0-bin-hadoop3/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "spark-3.5.0-bin-hadoop3/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 